# LangChain Hands-on 04 — Chains with LCEL & Runnables
**Trainer:** Ajeetkumar

**What we will practise**
- What is a chain? The `|` pipe operator
- prompt | llm | parser
- Sequential chains
- RunnableParallel, RunnablePassthrough, RunnableLambda
- Routing with RunnableBranch
- Fallbacks & retry
- Streaming and batching a whole chain

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

In [ ]:
! pip install -q langchain langchain-groq grandalf

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('groq_key')

In [ ]:
import os
os.environ['GROQ_API_KEY'] = groq_api_key

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b")

# What is a Chain?
- A chain = **steps joined together**: output of one step → input of the next.
- LCEL (LangChain Expression Language) uses the **pipe** `|` like Linux.

```
input dict ──▶ [Prompt] ──▶ [LLM] ──▶ [Parser] ──▶ final answer
```
- Every piece is a **Runnable** → has `invoke`, `stream`, `batch`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser

## 1. Without chain (3 manual steps)

In [ ]:
prompt = PromptTemplate.from_template("Explain {topic} in 3 lines")
formatted = prompt.format(topic="Blockchain")
response = llm.invoke(formatted)
answer = StrOutputParser().parse(response.content)
print(answer)

## 2. With chain (one line)

In [ ]:
chain = prompt | llm | StrOutputParser()

In [ ]:
print(chain.invoke({"topic": "Blockchain"}))

In [ ]:
print(chain.invoke({"topic": "Quantum computing"}))

In [ ]:
chain.get_graph().print_ascii()

## 3. Stream and batch the whole chain (free!)

In [ ]:
for chunk in chain.stream({"topic": "Black holes"}):
    print(chunk, end="", flush=True)

In [ ]:
answers = chain.batch([{"topic": "AI"}, {"topic": "IoT"}, {"topic": "5G"}])
for a in answers:
    print(a, "\n---")

## 4. Chain with structured output

In [ ]:
from pydantic import BaseModel

class Quiz(BaseModel):
    question: str
    options: list[str]
    answer: str

In [ ]:
quiz_prompt = ChatPromptTemplate.from_template("Create one MCQ on {topic} with 4 options.")
quiz_chain = quiz_prompt | llm.with_structured_output(Quiz)
quiz_chain.invoke({"topic": "Python lists"})

## 5. Sequential chain — output of chain 1 → input of chain 2

In [ ]:
title_chain = PromptTemplate.from_template("Give one catchy blog title about {topic}. Only the title.") | llm | StrOutputParser()
blog_chain = PromptTemplate.from_template("Write a 100-word blog intro for the title: {title}") | llm | StrOutputParser()

In [ ]:
full_chain = {"title": title_chain} | blog_chain
print(full_chain.invoke({"topic": "AI in Indian agriculture"}))

## 6. RunnablePassthrough — pass input as it is / add new keys

In [ ]:
from langchain_core.runnables import RunnablePassthrough

In [ ]:
RunnablePassthrough().invoke({"name": "Ajeet"})

In [ ]:
# .assign keeps the input AND adds a new key
chain = RunnablePassthrough.assign(title=title_chain)
chain.invoke({"topic": "Electric vehicles"})

## 7. RunnableParallel — run many chains at the same time

In [ ]:
from langchain_core.runnables import RunnableParallel

In [ ]:
summary_chain = PromptTemplate.from_template("Summarise in 1 line: {text}") | llm | StrOutputParser()
sentiment_chain = PromptTemplate.from_template("Sentiment of this text (positive/negative/neutral), one word: {text}") | llm | StrOutputParser()
keywords_chain = PromptTemplate.from_template("Give 3 keywords, comma separated: {text}") | llm | StrOutputParser()

In [ ]:
review_analyzer = RunnableParallel(summary=summary_chain, sentiment=sentiment_chain, keywords=keywords_chain)

In [ ]:
review = "The hotel room was clean and staff were very polite, but the WiFi kept disconnecting."
review_analyzer.invoke({"text": review})

## 8. RunnableLambda — use any Python function inside a chain

In [ ]:
from langchain_core.runnables import RunnableLambda

In [ ]:
def word_count(text):
    return {"text": text, "words": len(text.split())}

chain = PromptTemplate.from_template("Write a tweet about {topic}") | llm | StrOutputParser() | RunnableLambda(word_count)
chain.invoke({"topic": "monsoon"})

In [ ]:
# plain functions are converted automatically
chain = PromptTemplate.from_template("Write a tweet about {topic}") | llm | StrOutputParser() | (lambda t: t.upper())
chain.invoke({"topic": "chai"})

## 9. RunnableBranch — route to different chains (if / else)

In [ ]:
from langchain_core.runnables import RunnableBranch

In [ ]:
classify_chain = PromptTemplate.from_template(
    "Is this question about 'coding' or 'general'? One word.\nQuestion: {question}") | llm | StrOutputParser()

coding_chain = PromptTemplate.from_template("Answer with a short Python code example: {question}") | llm | StrOutputParser()
general_chain = PromptTemplate.from_template("Answer simply in 2 lines: {question}") | llm | StrOutputParser()

In [ ]:
router = RunnablePassthrough.assign(kind=classify_chain) | RunnableBranch(
    (lambda x: "coding" in x["kind"].lower(), coding_chain),
    general_chain,                       # default
)

In [ ]:
print(router.invoke({"question": "How to reverse a string in Python?"}))

In [ ]:
print(router.invoke({"question": "Why is the sky blue?"}))

## 10. Fallbacks & retry — production safety

In [ ]:
bad_llm = ChatGroq(model="model-that-does-not-exist")
safe_llm = bad_llm.with_fallbacks([llm])
print(safe_llm.invoke("Say: fallback worked").content)

In [ ]:
retry_llm = llm.with_retry(stop_after_attempt=3)
print(retry_llm.invoke("Say hi").content)

# Benefits of LCEL
- **Readable** pipelines with `|`
- **Streaming, batch, async for free**
- **Parallel** steps run at the same time → faster
- **Easy to debug** in LangSmith

# Limit
- A chain is a **fixed path**. If the LLM must decide "which tool next?" in a loop → use an **Agent** (notebook 13).

## 🧪 Practice
1. Chain: topic → 5 interview questions → answers for each (sequential).
2. Parallel chain: from a product name create a tagline, an Instagram caption and a tweet.
3. Router: send Hindi questions to a Hindi chain, English to an English chain.